# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NAJAM2005/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

> **Data note.** This notebook uses the public 30,000-page starter CSV in the repo (32 clients, 54.2% of pages labelled declining). The capstone uses a different, gated lane cut (10,550 pages, 29 clients, 45.0% declining, separate impression windows), so numbers here are not directly comparable with the capstone paper. Both are observed, directional and for decision-support.

**The queue, in words an editor can trust.** Pages are ranked by a score that combines staleness (more than 22 days since update) and demand (impressions). Each page gets one reason code and one action: STALE_HIGH_DEMAND -> REVIEW_FIRST, STALE_PAGE -> BATCH_REVIEW, HIGH_DEMAND_FRESH -> MONITOR, LOW_SIGNAL -> LOW_PRIORITY. As a check (the label is not in the score): REVIEW_FIRST pages decline 56.7% of the time, BATCH_REVIEW 62.3%, MONITOR 59.5% and LOW_PRIORITY 48.3%, against a 54.2% base rate, so the ordering is modest. **Warning:** 85.6% of REVIEW_FIRST pages and 80.8% of BATCH_REVIEW pages have exactly 104 days since update, a value shared by only 14 clients and held mostly by two, so a human must check what that value means before acting. The second table is the segment playbook from the capstone analysis (within-client demand tiers).

In [1]:
import os, json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
RAW = "https://raw.githubusercontent.com/NAJAM2005/flyrank-ml-internship/main/"
PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(PATH if os.path.exists(PATH) else RAW + PATH)      # public starter slice (anonymised)
df["is_declining"] = (df["trend_direction"] == "down").astype(int)   # outcome for checks only, never a feature
print(df.shape, "| declining share:", round(df["is_declining"].mean(), 3), "| clients:", df["client_id"].nunique())

df["stale"] = (df["days_since_last_update"] > 22).astype(float)
df["demand_pct"] = np.log1p(df["impressions_90d"]).rank(pct=True)
df["score"] = 0.6 * df["stale"] + 0.4 * df["demand_pct"]
hi = df["demand_pct"] >= 2 / 3
df["reason_code"] = np.select([(df["stale"] == 1) & hi, df["stale"] == 1, hi],
                              ["STALE_HIGH_DEMAND", "STALE_PAGE", "HIGH_DEMAND_FRESH"], "LOW_SIGNAL")
df["action"] = df["reason_code"].map({"STALE_HIGH_DEMAND": "REVIEW_FIRST", "STALE_PAGE": "BATCH_REVIEW",
                                      "HIGH_DEMAND_FRESH": "MONITOR", "LOW_SIGNAL": "LOW_PRIORITY"})
q = df.sort_values("score", ascending=False).reset_index(drop=True)
q["rank"] = q.index + 1
q["impr_band"] = pd.cut(q["impressions_90d"], [0, 1e3, 1e4, 1e5, 1e9], labels=["<1k", "1k-10k", "10k-100k", "100k+"])
print("Top 10 of the example queue (no page identifiers shown):")
print(q.head(10)[["rank", "action", "reason_code", "days_since_last_update", "impr_band"]].to_string(index=False))
print("\nactions across all pages, with the observed declining share (a check only; the label is not in the score):")
chk = q.groupby("action").agg(pages=("is_declining", "size"), declining_share=("is_declining", "mean"),
                              share_at_104_days=("days_since_last_update", lambda x: (x == 104).mean())).round(3)
print(chk.sort_values("pages", ascending=False).to_string())
print("base rate:", round(df["is_declining"].mean(), 3))

playbook = pd.DataFrame([
    [1, "181-365 days, high demand", "REVIEW_FIRST", "54.7% decline; 24.9% of lost impressions"],
    [2, "91-180 days, high demand", "REVIEW_FIRST (by value)", "average risk, 32.4% of lost impressions"],
    [3, "181-365 days, low demand", "BATCH_REVIEW", "53.3% decline, only 2.4% of loss"],
    [4, "365+ days, high demand", "MONITOR", "share of loss halves without the top client"],
    [5, "31-90 days, any demand", "LOW_PRIORITY", "17-24 points below base rate"]],
    columns=["priority", "segment", "action", "evidence (capstone, within-client tiers)"])
print("\nPlaybook from the capstone analysis:")
print(playbook.to_string(index=False))


(30000, 45) | declining share: 0.542 | clients: 32
Top 10 of the example queue (no page identifiers shown):
 rank       action       reason_code  days_since_last_update impr_band
    1 REVIEW_FIRST STALE_HIGH_DEMAND                     104     100k+
    2 REVIEW_FIRST STALE_HIGH_DEMAND                      48     100k+
    3 REVIEW_FIRST STALE_HIGH_DEMAND                     104     100k+
    4 REVIEW_FIRST STALE_HIGH_DEMAND                     104     100k+
    5 REVIEW_FIRST STALE_HIGH_DEMAND                     104     100k+
    6 REVIEW_FIRST STALE_HIGH_DEMAND                     104     100k+
    7 REVIEW_FIRST STALE_HIGH_DEMAND                     104     100k+
    8 REVIEW_FIRST STALE_HIGH_DEMAND                     104     100k+
    9 REVIEW_FIRST STALE_HIGH_DEMAND                     104     100k+
   10 REVIEW_FIRST STALE_HIGH_DEMAND                     104     100k+

actions across all pages, with the observed declining share (a check only; the label is not in the score):
   

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use and limits.** For an SEO editor choosing which pages of one client to open first. It is valid for pages like those analysed (10,550 pages, 29 clients, one June-2026 snapshot). The evidence is weak (per-client AUC 0.567 for the model, 0.561 for the rule) and most of the value comes from page size: ranking by impressions finds most lost impressions. It stops being valid for a different time period, a new type of client, or pages outside this lane. It does not show that a refresh changes traffic.

In [2]:
cap = json.loads(pd.io.common.urlopen(RAW + "work/outputs/capstone_metrics.json").read())
use = pd.DataFrame([
    ["who uses it", "an SEO editor deciding which pages of one client to open first"],
    ["valid for", f"pages like the {cap['data']['pages_analysed']:,} analysed ({cap['data']['clients']} clients, one June-2026 snapshot)"],
    ["strength of evidence", f"weak: mean per-client AUC {cap['mean_client_auc']['old_window+age+static']:.3f} for the model, "
                             f"{cap['mean_client_auc']['baseline']:.3f} for the rule"],
    ["what drives the value", "page size: ranking by impressions finds most lost impressions"],
    ["stops being valid when", "a different time period, a new client type, or pages outside this lane"]],
    columns=["item", "statement"])
pd.set_option("display.max_colwidth", 140)
print(use.to_string(index=False))


                  item                                                              statement
           who uses it         an SEO editor deciding which pages of one client to open first
             valid for    pages like the 10,550 analysed (29 clients, one June-2026 snapshot)
  strength of evidence      weak: mean per-client AUC 0.567 for the model, 0.561 for the rule
 what drives the value          page size: ranking by impressions finds most lost impressions
stops being valid when a different time period, a new client type, or pages outside this lane


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human review and the no-go list.** Before acting, a person opens the page and checks that it is not already being rewritten or recovering, that the impressions come from relevant queries, and that the drop is not a seasonal dip. Never automate: rewriting, publishing or deleting a page from a score alone; telling a client a page "will" decline or that a refresh "will" lift traffic; or treating reason codes as proof of cause.

In [3]:
review = pd.DataFrame([
    ["before acting", "open the page and check it is not already being rewritten or recovering"],
    ["before acting", "check the impressions come from relevant queries, not a few odd ones"],
    ["before acting", "check the decline is not a seasonal dip seen in the same month last year"],
    ["never automate", "rewriting, publishing or deleting a page from a score alone"],
    ["never automate", "telling a client a page 'will' decline or that a refresh 'will' lift traffic"],
    ["never automate", "using reason codes as proof of cause"]],
    columns=["type", "rule"])
print(review.to_string(index=False))


          type                                                                         rule
 before acting      open the page and check it is not already being rewritten or recovering
 before acting         check the impressions come from relevant queries, not a few odd ones
 before acting     check the decline is not a seasonal dip seen in the same month last year
never automate                  rewriting, publishing or deleting a page from a score alone
never automate telling a client a page 'will' decline or that a refresh 'will' lift traffic
never automate                                         using reason codes as proof of cause


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Monitoring and retrain triggers.** The recommendations have gone stale if the label rate leaves 35-55%, if the rule's per-client AUC falls below 0.5 in 3 or more of 10 clients, if the top 20% of pages finds under 40% of lost impressions (now 50.9% for the rule), if a new dated snapshot or warehouse release becomes available, or if one client holds more than half of the lost impressions (now 46.9%).

In [4]:
triggers = pd.DataFrame([
    ["label rate", f"moves outside 35%-55% (analysed sample: {cap['data']['label_rate']:.1%})", "re-check the label and thresholds"],
    ["rule vs random", "per-client AUC of the rule falls below 0.5 in 3 or more of 10 clients", "stop using the rule for ranking"],
    ["lost-impression capture", "top 20% of pages finds under 40% of lost impressions (now 50.9% for the rule)", "re-tune the rule"],
    ["new data", "a new snapshot or a dated warehouse release becomes available", "rebuild features and re-run the comparison"],
    ["concentration", "one client holds over 50% of lost impressions (now 46.9%)", "report per client, not pooled"]],
    columns=["watch", "trigger", "response"])
pd.set_option("display.max_colwidth", 120)
print(triggers.to_string(index=False))


                  watch                                                                       trigger                                   response
             label rate                                moves outside 35%-55% (analysed sample: 45.0%)          re-check the label and thresholds
         rule vs random         per-client AUC of the rule falls below 0.5 in 3 or more of 10 clients            stop using the rule for ranking
lost-impression capture top 20% of pages finds under 40% of lost impressions (now 50.9% for the rule)                           re-tune the rule
               new data                 a new snapshot or a dated warehouse release becomes available rebuild features and re-run the comparison
          concentration                     one client holds over 50% of lost impressions (now 46.9%)              report per client, not pooled


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

**Exports for the paper.** The cell writes `work/outputs/action_playbook.json` (playbook, review rules, monitoring triggers and action counts). It contains no page or client identifiers (the check below counts them). Data files such as CSVs are not committed; this JSON is.

In [5]:
os.makedirs("work/outputs", exist_ok=True)
export = {"purpose": "decision-support triage list; not a prediction of individual pages",
          "playbook": playbook.to_dict(orient="records"),
          "review_rules": review.to_dict(orient="records"),
          "monitoring_triggers": triggers.to_dict(orient="records"),
          "example_queue_size": int(len(q)), "actions": q["action"].value_counts().to_dict()}
with open("work/outputs/action_playbook.json", "w") as f:
    json.dump(export, f, indent=2)
txt = open("work/outputs/action_playbook.json").read()
import re
print("wrote work/outputs/action_playbook.json,", len(txt), "bytes")
print("identifier-like strings in the export:", len(re.findall(r"(?:client|content)_[0-9a-f]{6,}", txt)))


wrote work/outputs/action_playbook.json, 2965 bytes
identifier-like strings in the export: 0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.